# 🎯 Aykırı Değer (Outlier) Analizi ve Yönetimi

Bu notebook, IQR, Z-Score ve Isolation Forest gibi ileri seviye teknikleri barındırır.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

## 1. IQR Yöntemi (Interquartile Range)

In [ ]:
def detect_outliers_iqr(df, column, threshold=1.5):
    """
    Bir sütundaki aykırı değerlerin indekslerini IQR yöntemiyle tespit eder.
    """
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - threshold * IQR
    upper_bound = Q3 + threshold * IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers.index

def cap_outliers_iqr(df, column, threshold=1.5):
    """
    Aykırı değerleri silmek yerine, alt ve üst sınırlara baskılar (Capping/Winsorization).
    """
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - threshold * IQR
    upper_bound = Q3 + threshold * IQR
    
    df_capped = df.copy()
    df_capped[column] = np.where(df_capped[column] < lower_bound, lower_bound, df_capped[column])
    df_capped[column] = np.where(df_capped[column] > upper_bound, upper_bound, df_capped[column])
    
    return df_capped

## 2. Z-Score Yöntemi
Normal dağılıma sahip (veya çok yakın) verilerde 3 standart sapma ötesini bulur.

In [ ]:
from scipy import stats

def detect_outliers_zscore(df, column, threshold=3.0):
    z_scores = np.abs(stats.zscore(df[column].dropna()))
    outliers = df.dropna(subset=[column])[z_scores > threshold]
    return outliers.index

## 3. Profesyonel: Isolation Forest (Makine Öğrenmesi ile)
Çok boyutlu veri setlerinde aykırı değerleri bulmak için tüm sütunlara birden bakar.

In [ ]:
def detect_outliers_isolation_forest(df, columns_to_use, contamination=0.01):
    """
    contamination: Tahmini aykırı değer oranı (örn. %1)
    """
    iso = IsolationForest(contamination=contamination, random_state=42)
    
    # Eksik değerleri geçici olarak medyan ile doldurup tahmine sokuyoruz
    temp_df = df[columns_to_use].fillna(df[columns_to_use].median())
    
    preds = iso.fit_predict(temp_df)
    
    # -1 olanlar aykırı değerdir
    outlier_indices = df[preds == -1].index
    return outlier_indices

# Örnek Kullanım:
# numeric_cols = df_train.select_dtypes(include=[np.number]).columns
# outliers = detect_outliers_isolation_forest(df_train, numeric_cols, contamination=0.01)
# print(f'{len(outliers)} adet aykırı satır bulundu.')
# df_train = df_train.drop(outliers).reset_index(drop=True)